# 과제 1: Voting vs Stacking 비교  
California Housing 데이터로 회귀 문제에서 Voting과 Stacking 비교하기  
- 데이터: sklearn.datasets.fetch_california_housing()  
- VotingRegressor와 StackingRegressor 사용  
- Base Models: 최소 4개 이상의 회귀 모델  
- Hard Voting, Soft Voting, Stacking 성능 비교  
- 여러가지 평가모델로 평가  

In [15]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import VotingRegressor, StackingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

In [16]:
# 데이터 준비
X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 베이스 모델
base_models = [
    ('ridge', Ridge(alpha=1.0)),
    ('lasso', Lasso(alpha=0.1)),
    ('svr', make_pipeline(StandardScaler(), SVR(C=1.0, epsilon=0.1))),
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('xgb', xgb.XGBRegressor(n_estimators=100, random_state=42)),
]

# 앙상블 모델
ensemble_models = [
    ('voting', VotingRegressor(estimators=base_models, weights=[0.5, 0.5, 3, 0.5, 0.1])),
    ('stacking', StackingRegressor(estimators=base_models, final_estimator=LinearRegression(), cv=5))
]

all_models = base_models + ensemble_models


# 평가 로직
def calculate_metrics(y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return [mae, mse, rmse, r2]

results = []

for name, model in all_models:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics = calculate_metrics(y_test, y_pred)
    results.append([name] + metrics)

# 결과 출력
df_columns = ['Model', 'MAE', 'MSE', 'RMSE', 'R2']
results_df = pd.DataFrame(results, columns=df_columns)
display(results_df.round(4))

,Model,MAE,MSE,RMSE,R2
0,ridge,0.5332,0.5558,0.7455,0.5759
1,lasso,0.5816,0.6135,0.7833,0.5318
2,svr,0.3986,0.3570,0.5975,0.7276
3,rf,0.3276,0.2557,0.5057,0.8049
4,xgb,0.3096,0.2226,0.4718,0.8301
5,voting,0.3949,0.3454,0.5877,0.7364
6,stacking,0.3040,0.2169,0.4658,0.8345


---
# 과제 2: Stacking 하이퍼파라미터 튜닝  
Breast Cancer 데이터로 최적의 Stacking 모델 찾기  

- Base Models 3-5개 선택
- 각 Base Model의 주요 파라미터 튜닝
- Meta Model 선택 (Logistic Regression, Random Forest 등)
- passthrough 옵션 비교
- 최종 모델의 Confusion Matrix, Classification Report 출력

In [24]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

In [22]:
# 데이터 준비
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 베이스 모델
base_models = [
    ('logi', LogisticRegression(max_iter=10000, random_state=42)),
    ('svc', SVC(probability=True, random_state=42)),
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42))
]

# Stacking 모델
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=xgb.XGBClassifier(random_state=42),
    cv=5
)

# 하이퍼파라미터 그리드
param_gird = {
    'logi__solver' : ['lbfgs', 'saga'],
    'svc__C' : [0.1, 1, 10],
    'dt__max_depth' : [3, 5, 7],
    'rf__n_estimators' : [50, 100],
    'cv' : [3, 5],
    'passthrough' : [True, False],
}

gs = GridSearchCV(
    stacking_clf,
    param_grid=param_gird,
    cv=3,
    n_jobs=-1,
    scoring='accuracy'
)

gs.fit(X_train, y_train)
y_pred = gs.best_estimator_.predict(X_test)

# 결과 출력
print('='*30 + ' Classification Report ' + '='*30)
print(classification_report(y_test, y_pred))

============================== Classification Report ==============================
              precision    recall  f1-score   support

           0       1.00      0.93      0.96        43
           1       0.96      1.00      0.98        71

    accuracy                           0.97       114
   macro avg       0.98      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114



---
# 과제 3: 커스텀 Voting 가중치 최적화  
GridSearchCV를 사용하여 최적의 Voting 가중치 찾기

- 5개 이상의 다양한 분류 모델 사용
- 가중치를 파라미터로 설정하여 GridSearch
- 최적 가중치 조합 찾기

In [26]:
voting_clf_weight = VotingClassifier(
    estimators=base_models,
    voting='soft'
)

param_grid = {
    'weights': [
        [1, 1, 1, 1],
        [2, 1, 1, 1],
        [1, 2, 1, 1],
        [1, 1, 2, 1],
        [1, 1, 1, 2],
        [2, 2, 1, 1],
        [2, 1, 2, 1],
        [2, 1, 1, 2],
        [1, 2, 2, 1],
        [1, 2, 1, 2],
        [1, 1, 2, 2],
    ]
}

grid_search = GridSearchCV(
    voting_clf_weight, 
    param_grid, 
    cv=5,
    scoring='accuracy',
    n_jobs=1,
    verbose=1
)

grid_search.fit(X_train, y_train)

# 결과 출력
print("="*60)
print("최적 가중치:", grid_search.best_params_['weights'])
print(f"최적 CV 점수: {grid_search.best_score_:.4f}")
print(f"테스트 점수: {grid_search.score(X_test, y_test):.4f}")

print("="*60)
print("가중치별 성능 (상위 5개):")

# 결과를 정렬하여 출력
results = grid_search.cv_results_
sorted_indices = results['rank_test_score'].argsort()

for i in sorted_indices[:5]:
    weights = param_grid['weights'][i]
    mean_score = results['mean_test_score'][i]
    std_score = results['std_test_score'][i]
    print(f"{weights}: {mean_score:.4f} (+/- {std_score:.4f})")

Fitting 5 folds for each of 11 candidates, totalling 55 fits
최적 가중치: [2, 1, 1, 1]
최적 CV 점수: 0.9626
테스트 점수: 0.9649
가중치별 성능 (상위 5개):
[2, 1, 1, 1]: 0.9626 (+/- 0.0192)
[1, 1, 1, 2]: 0.9560 (+/- 0.0197)
[2, 2, 1, 1]: 0.9560 (+/- 0.0251)
[2, 1, 1, 2]: 0.9560 (+/- 0.0155)
[1, 2, 1, 1]: 0.9516 (+/- 0.0266)
